# Paper figures

Each figure is built from cached per-model artifacts under `$ANALYSIS_DIR/<model>/`.

- **Models** are defined once in `conf/analysis/models.yaml`; the `MODELS` list below picks which ones (and in what order) to compare.
- **Extraction** happens in SLURM jobs, not here. Run the *preflight* cell to see what is missing and submit `extract.sbatch` jobs; come back once they finish.
- Each figure cell returns a `Figure` shown inline and saved to `report/figures/<name>.pdf`.

Kernel: the project `.venv`.

In [1]:
import sys, pathlib
REPO = pathlib.Path.cwd()
while not (REPO / 'pyproject.toml').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

import matplotlib
matplotlib.use('Agg')  # non-interactive: figures only render when explicitly displayed

from IPython.display import display
from protein_design.wandb_plots import set_publication_style
from protein_design.analysis import figures, registry
set_publication_style()

# --- the models to compare (keys from conf/analysis/models.yaml) ---
# Add / remove / reorder freely; order is the plotting order.
MODELS = [
    'vanilla_35m',
    'evo_35m',
    'evo_c05_blosum25',
    'vanilla_650m',
    'evo_650m',
    'just_dpo_4ep_step1376',
]

# Datasets to use (a list of keys, or 'all' for every dataset in the registry).
DATASETS = 'all'

print('known models:', sorted(registry.load_models()))
print('known datasets:', registry.dataset_keys('all'))

known models: ['c05_blosum25_seed', 'evo_35m', 'evo_650m', 'evo_c05_blosum25', 'evo_dedup_35m', 'just_dpo_4ep_step1376', 'vanilla_35m', 'vanilla_650m']
known datasets: ['ed2_m22', 'ed5_m22', 'ed811_m22', 'ed2_si06', 'ed5_si06', 'exp']


## Preflight — what still needs extracting

Prints the exact `sbatch` commands for any missing artifacts. Run them in a terminal (or submit from here with `subprocess`), then re-run the figure cells once the jobs finish.

In [2]:
missing = figures.preflight(MODELS, DATASETS, kind='pll')
# Copy the sbatch commands above into a terminal to submit the extraction jobs.
# Return here and re-run the figure cells once the jobs finish.

All pll artifacts present for 6 model(s) x 6 dataset(s).


## Figure: PLL vs experimental enrichment (Spearman)

Grouped bars — one bar per model per dataset. Higher = the model's CDR-H3 pseudo-log-likelihood ranks experimental binding enrichment better.

In [ ]:
fig = figures.plot_pll_spearman(MODELS, DATASETS)
out = figures.save_fig(fig, 'pll_spearman')  # prints: Saved: <path>
display(fig)

Saved: /cluster/home/mdenegri/protein-design/report/figures/pll_spearman.pdf


<Figure size 1230x630 with 1 Axes>

## Preflight — scorer predictions

Scorer predictions (M22/SI06 binder scorer) are computed once per dataset with `score_dms.sbatch` (uses the `flu` conda env). They don't vary per model — run this only if the scorer CSV is missing.

In [5]:
missing_scorer = figures.preflight_scorer(DATASETS)
# Uncomment to submit:
# import subprocess
# for d in missing_scorer:
#     subprocess.run(["sbatch", "bash_scripts/utils/score_dms.sbatch", "--dataset", d], check=True)

All scorer artifacts present.


## Figure: PLL vs supervised scorer (Spearman)

Grouped bars show Spearman(PLL, truth) per model per dataset — same as the figure above. The **dashed horizontal segment** per dataset is Spearman(scorer, truth): the fixed binder scorer's correlation with experimental truth. This is the supervised upper bound; bars above the dashed line mean the model's PLL is a better ranker than the scorer.

In [6]:
fig = figures.plot_pll_vs_scorer_spearman(MODELS, DATASETS)
figures.save_fig(fig, 'pll_vs_scorer_spearman')
fig

Saved: /cluster/home/mdenegri/protein-design/report/figures/pll_vs_scorer_spearman.pdf


<Figure size 1230x630 with 1 Axes>

## Figure: CDR-H3 pseudo-perplexity

Per-sequence pseudo-perplexity PP = exp(−PLL / L), where L is the CDR-H3 length. **Lower PP = model assigns higher average probability to each residue.** Sequences are pooled across the selected datasets (deduplicated by sequence string).

In [7]:
fig = figures.plot_pseudo_perplexity(MODELS, DATASETS)
figures.save_fig(fig, 'pseudo_perplexity')
fig

Saved: /cluster/home/mdenegri/protein-design/report/figures/pseudo_perplexity.pdf


<Figure size 1080x600 with 1 Axes>